In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from scipy.stats import kendalltau

# ==========================================
# 1. The Deep Learning Architecture
# ==========================================
class NarrativeTransformer(nn.Module):
    def __init__(self, embed_dim=1024, max_pages=100, num_heads=8, num_layers=4):
        super(NarrativeTransformer, self).__init__()
        
        # We project the embedding if needed, but here we assume bge-large (1024)
        self.embed_dim = embed_dim
        
        # The Transformer Encoder allows all shuffled pages to "talk" to each other
        # and figure out where they belong relative to the whole pile.
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, 
            nhead=num_heads, 
            dim_feedforward=2048, 
            dropout=0.1, 
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # The output layer guesses the absolute position (0 to max_pages-1)
        self.position_predictor = nn.Linear(embed_dim, max_pages)

    def forward(self, x):
        # x shape: [Batch_Size, Num_Pages, Embed_Dim]
        contextualized_features = self.transformer(x)
        
        # logits shape: [Batch_Size, Num_Pages, Max_Pages]
        # For each page, it outputs a score for every possible position
        logits = self.position_predictor(contextualized_features)
        return logits

# ==========================================
# 2. Data Preparation for PyTorch
# ==========================================
def prepare_training_data(train_csv_path, embed_model_name="BAAI/bge-large-en-v1.5", num_epochs=200):
    print("Loading data and embedding model...")
    df = pd.read_csv(train_csv_path)
    # Ensure it's sorted perfectly for the ground truth
    df = df.sort_values(by='page').reset_index(drop=True) 
    
    embed_model = SentenceTransformer(embed_model_name)
    print("Generating dense embeddings...")
    embeddings = embed_model.encode(df['text'].tolist(), convert_to_numpy=True)
    
    num_pages = len(embeddings)
    embed_dim = embeddings.shape[1]
    
    # We will dynamically generate shuffled batches during training
    return embeddings, num_pages, embed_dim

def get_shuffled_batch(base_embeddings, batch_size=16):
    """
    Creates a batch of shuffled books. 
    The target is the original index of each shuffled page.
    """
    num_pages = len(base_embeddings)
    embed_dim = base_embeddings.shape[1]
    
    batch_x = torch.zeros((batch_size, num_pages, embed_dim))
    batch_y = torch.zeros((batch_size, num_pages), dtype=torch.long)
    
    for b in range(batch_size):
        # Create a random permutation
        perm = torch.randperm(num_pages)
        batch_x[b] = torch.tensor(base_embeddings[perm])
        
        # The target is the inverse of the permutation (where the page SHOULD go)
        # If perm[0] = 45, it means the 0th shuffled page is actually page 45.
        batch_y[b] = perm
        
    return batch_x.cuda(), batch_y.cuda()

# ==========================================
# 3. Training Loop
# ==========================================
def train_model(embeddings, num_pages, embed_dim, epochs=300):
    print("Initializing Narrative Transformer...")
    model = NarrativeTransformer(embed_dim=embed_dim, max_pages=num_pages).cuda()
    
    # CrossEntropyLoss is perfect here because we treat each position (1 to N) as a "Class"
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=10, factor=0.5)
    
    print("Starting Training...")
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        
        # Train on 16 differently shuffled versions of the book at once
        x, targets = get_shuffled_batch(embeddings, batch_size=16)
        
        logits = model(x) # Shape: [16, 100, 100]
        
        # Reshape for CrossEntropyLoss
        # Logits must be [Batch * Num_Pages, Num_Classes]
        logits_flat = logits.view(-1, num_pages)
        targets_flat = targets.view(-1)
        
        loss = criterion(logits_flat, targets_flat)
        loss.backward()
        optimizer.step()
        
        if epoch % 20 == 0:
            print(f"Epoch {epoch:03d} | Loss: {loss.item():.4f}")
            
        scheduler.step(loss)
        
    print("Training Complete.")
    return model

# ==========================================
# 4. Inference and Sequence Extraction
# ==========================================
def solve_book(model, test_csv_path, embed_model_name="BAAI/bge-large-en-v1.5"):
    df = pd.read_csv(test_csv_path)
    embed_model = SentenceTransformer(embed_model_name)
    
    embeddings = embed_model.encode(df['text'].tolist(), convert_to_numpy=True)
    x = torch.tensor(embeddings).unsqueeze(0).cuda() # Add batch dimension [1, 100, 1024]
    
    model.eval()
    with torch.no_grad():
        logits = model(x).squeeze(0) # Shape: [100, 100]
        
    # logits[i, j] is the score that the i-th shuffled page belongs in the j-th position
    # To get a strict 1-to-1 mapping, we can use an argmax sort or scipy's linear_sum_assignment
    from scipy.optimize import linear_sum_assignment
    
    # Convert logits to a cost matrix (lower cost is better)
    cost_matrix = -logits.cpu().numpy() 
    
    # linear_sum_assignment guarantees no two pages get assigned to the same position
    shuffled_indices, predicted_positions = linear_sum_assignment(cost_matrix)
    
    # Order the pages based on the predicted positions
    ordered_page_ids = [df.iloc[i]['page'] for i in np.argsort(predicted_positions)]
    
    return ordered_page_ids

# ==========================================
# EXECUTION
# ==========================================
if __name__ == "__main__":
    # 1. Prepare and Train
    embeddings, num_pages, embed_dim = prepare_training_data("./data/Mysterious_Affair_at_Styles_Train_Data.csv")
    trained_model = train_model(embeddings, num_pages, embed_dim, epochs=300)
    
    # 2. Solve Test Books
    print("Solving Test Books...")
    book_a_order = solve_book(trained_model, "./data/BookA_test.csv")
    book_b_order = solve_book(trained_model, "./data/BookB_test.csv")
    
    # 3. Save to CSV
    pd.DataFrame({"original_page": range(1, len(book_a_order)+1), "shuffled_page": book_a_order}).to_csv("BookA_Final.csv", index=False)
    pd.DataFrame({"original_page": range(1, len(book_b_order)+1), "shuffled_page": book_b_order}).to_csv("BookB_Final.csv", index=False)
    print("Complete!")

Loading data and embedding model...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating dense embeddings...
Initializing Narrative Transformer...
Starting Training...
Epoch 000 | Loss: 5.4744
Epoch 020 | Loss: 0.4483
Epoch 040 | Loss: 0.0217
Epoch 060 | Loss: 0.0096
Epoch 080 | Loss: 0.0070
Epoch 100 | Loss: 0.0057
Epoch 120 | Loss: 0.0048
Epoch 140 | Loss: 0.0041
Epoch 160 | Loss: 0.0035
Epoch 180 | Loss: 0.0031
Epoch 200 | Loss: 0.0027
Epoch 220 | Loss: 0.0024
Epoch 240 | Loss: 0.0022
Epoch 260 | Loss: 0.0019
Epoch 280 | Loss: 0.0017
Training Complete.
Solving Test Books...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Complete!


In [ ]:
import numpy as np
import pandas as pd
import random
import math
from sentence_transformers.cross_encoder import CrossEncoder

# Load your fine-tuned Cross-Encoder
CE_MODEL = CrossEncoder('./fine-tuned-cross-encoder')

def calculate_sequence_score(sequence, tails, heads, pos_scores, alpha=0.7):
    """
    Calculates the 'Energy' or 'Fitness' of a sequence.
    Higher is better.
    """
    score = 0.0
    N = len(sequence)
    
    # 1. Local Score: Add up all the Cross-Encoder probabilities for adjacent pages
    for i in range(N - 1):
        page_a_idx = sequence[i]
        page_b_idx = sequence[i + 1]
        
        # Cross-Encoder predicts P(B comes after A)
        ce_prob = CE_MODEL.predict([tails[page_a_idx], heads[page_b_idx]])
        score += ce_prob * alpha
        
    # 2. Global Score: Penalize pages that are far from their Ridge Regression predicted absolute position
    for i in range(N):
        current_page_idx = sequence[i]
        actual_fractional_pos = i / N
        predicted_fractional_pos = pos_scores[current_page_idx]
        
        # Deduct points if a page is placed far from where RidgeCV thinks it belongs
        penalty = abs(actual_fractional_pos - predicted_fractional_pos)
        score -= penalty * (1.0 - alpha)
        
    return float(score)

def simulated_annealing_sort(rep: dict, features: np.ndarray, initial_temp=10.0, cooling_rate=0.995, iterations=5000):
    """
    Uses Stochastic Local Search to iteratively untangle the narrative.
    """
    print("Starting Simulated Annealing Optimization...")
    N = len(features)
    tails = rep["tails"]
    heads = rep["heads"]
    
    # 1. The Seed: Start with your highly accurate 0.64 Ridge Regression sequence
    pos_scores = position_model.predict(features)
    current_sequence = list(np.argsort(pos_scores, kind="mergesort"))
    
    # Calculate the baseline score
    current_score = calculate_sequence_score(current_sequence, tails, heads, pos_scores)
    
    best_sequence = current_sequence.copy()
    best_score = current_score
    
    temp = initial_temp
    
    for i in range(iterations):
        # Create a new candidate by swapping two random adjacent or near-adjacent pages
        candidate_sequence = current_sequence.copy()
        
        # 80% chance to swap adjacent pages (fixing micro-errors)
        # 20% chance to swap random pages (fixing macro-errors)
        if random.random() < 0.8:
            idx1 = random.randint(0, N - 2)
            idx2 = idx1 + 1
        else:
            idx1 = random.randint(0, N - 1)
            idx2 = random.randint(0, N - 1)
            
        # Execute the swap
        candidate_sequence[idx1], candidate_sequence[idx2] = candidate_sequence[idx2], candidate_sequence[idx1]
        
        # Score the new candidate
        candidate_score = calculate_sequence_score(candidate_sequence, tails, heads, pos_scores)
        
        # Did it improve?
        if candidate_score > current_score:
            current_sequence = candidate_sequence
            current_score = candidate_score
            if candidate_score > best_score:
                best_sequence = candidate_sequence.copy()
                best_score = candidate_score
        else:
            # If it got worse, sometimes accept it anyway to escape local optima
            # (This is the magic of Simulated Annealing)
            acceptance_probability = math.exp((candidate_score - current_score) / temp)
            if random.random() < acceptance_probability:
                current_sequence = candidate_sequence
                current_score = candidate_score
                
        # Cool the temperature down
        temp *= cooling_rate
        
        if i % 1000 == 0:
            print(f"Iteration {i}/{iterations} | Current Best Score: {best_score:.4f} | Temp: {temp:.4f}")
            
    print("Simulated Annealing Complete!")
    return best_sequence

# ==========================================
# How to use it in your execution block:
# ==========================================
def save_submission(frame: pd.DataFrame, rep: dict, features: np.ndarray, out_path):
    # Call the new annealing function instead of TSP
    order = simulated_annealing_sort(rep, features, iterations=8000) 
    
    submission = pd.DataFrame({
        "original_page": np.arange(1, len(order) + 1, dtype=int),
        "shuffled_page": frame.iloc[order]["page"].to_numpy(dtype=int),
    })
    submission.to_csv(out_path, index=False)
    return order

save_submission